In [ ]:
from __future__ import annotations

import re
from pathlib import Path
from urllib.request import urlretrieve

import anndata as ad
import numpy as np
import pandas as pd

%matplotlib inline

import sys

sys.path.append(r"R:\Dante\git\cc_mapping\src")

# Downloading data

In [ ]:
url = "https://zenodo.org/records/4525425/files/control_manifold_allfeatures.csv?download=1"

cwd = Path.cwd()

download_path = cwd / "data" / "control_manifold_allfeatures.csv"

if not download_path.parent.exists():
    download_path.parent.mkdir(parents=True)
    urlretrieve(url, download_path)

# Loading in data

In [ ]:
csv = pd.read_csv(download_path, index_col=0, low_memory=False)
csv.index = csv.index.astype(str)
csv.drop(["annotated age", "annotated phase"], axis=1, inplace=True)
csv.head()

# CSV Preprocessing

The csv file needs to eventually converted into an Anndata object. This requires us to process the data for this process

In [ ]:
# initial columns
csv.columns

In [ ]:
import math
import statistics
from collections import defaultdict

import Levenshtein
from scipy.cluster.hierarchy import fcluster, linkage


# --- Helper function for distance (unchanged) ---
def calculate_normalized_distance(s1, s2):
    """Helper function to calculate normalized Levenshtein distance."""
    if not s1 and not s2:
        return 0.0
    if not s1 or not s2:
        return 1.0

    dist = Levenshtein.distance(s1, s2)
    max_len = max(len(s1), len(s2))

    if max_len == 0:
        norm_dist = 0.0
    else:
        norm_dist = dist / max_len
        if not math.isfinite(norm_dist):
            norm_dist = 1.0
    return norm_dist


# --- Helper function for internal sorting (unchanged, outliers first) ---
def sort_items_within_cluster(cluster: list[str]) -> list[str]:
    """
    Sorts items within a cluster based on avg distance to other members.
    Items LEAST similar to peers (highest avg distance) will appear FIRST.
    """
    if len(cluster) <= 1:
        return cluster

    item_distances = []
    for i, item in enumerate(cluster):
        distances_to_others = []
        for j, other_item in enumerate(cluster):
            if i == j:
                continue
            # Use the helper for consistency
            distances_to_others.append(calculate_normalized_distance(item, other_item))

        avg_dist = statistics.mean(distances_to_others) if distances_to_others else 0.0
        item_distances.append({"item": item, "avg_dist": avg_dist})

    # Sort items based on their average distance (DESCENDING) -> outliers first
    sorted_items_with_dist = sorted(item_distances, key=lambda x: -x["avg_dist"])

    return [d["item"] for d in sorted_items_with_dist]


# --- Main function using SciPy Hierarchical Clustering ---
def group_columns_hierarchical_internal_sort(
    column_names: list[str],
    normalized_threshold: float = 0.4,  # Threshold for fcluster distance criterion
    linkage_method: str = "average",  # Method for scipy.cluster.hierarchy.linkage
    sort_clusters_by: str = "size",  # Options: 'size', 'none'
) -> list[list[str]]:
    """
    Groups similar column names using hierarchical clustering (SciPy),
    sorts items within each cluster (outliers first), and optionally
    sorts the list of clusters themselves.

    Args:
        column_names: A list of column name strings.
        normalized_threshold: The maximum normalized distance criterion for forming
                              flat clusters using fcluster. Lower = stricter.
        linkage_method: The linkage algorithm for hierarchical clustering
                        (e.g., 'average', 'complete', 'ward', 'single').
        sort_clusters_by: How to sort the final list of clusters.
                 'size': Sort clusters by number of members (descending). Largest first.
                 'none': Keep the order determined by fcluster (often arbitrary).

    Returns:
        A list of lists (clusters). Clusters are optionally sorted based on
        sort_clusters_by. Items WITHIN each cluster are always sorted with the
        least representative items (outliers) first.
    """
    if not column_names:
        return []

    # Ensure unique names first, remember original order/duplicates if needed (not done here)
    # Using unique names for clustering is generally better
    unique_column_names = sorted(
        list(set(column_names))
    )  # Use unique names for clustering
    if len(unique_column_names) <= 1:
        # If only one unique name, just apply internal sort (which does nothing)
        return [
            sort_items_within_cluster(list(column_names))
        ]  # Return original list structure

    num_unique_cols = len(unique_column_names)

    # --- 1. Calculate Condensed Distance Matrix ---
    # Required format for scipy.cluster.hierarchy.linkage is a 1D array
    condensed_dist_matrix = np.zeros((num_unique_cols * (num_unique_cols - 1)) // 2)
    k = 0
    for i in range(num_unique_cols):
        for j in range(i + 1, num_unique_cols):
            dist = calculate_normalized_distance(
                unique_column_names[i], unique_column_names[j]
            )
            condensed_dist_matrix[k] = dist
            k += 1

    # --- 2. Perform Hierarchical Clustering ---
    if condensed_dist_matrix.size == 0:
        # Handle edge case where all columns might be identical after unique()
        cluster_labels = np.ones(num_unique_cols, dtype=int)
    else:
        linked = linkage(condensed_dist_matrix, method=linkage_method)

        # --- 3. Form Flat Clusters ---
        # 'fcluster' creates flat clusters from the hierarchical clustering.
        # 'criterion='distance'' cuts the dendrogram at the specified distance threshold.
        cluster_labels = fcluster(linked, normalized_threshold, criterion="distance")

    # --- 4. Group Unique Names by Cluster Label ---
    temp_grouped_clusters = defaultdict(list)
    for i, label in enumerate(cluster_labels):
        temp_grouped_clusters[label].append(unique_column_names[i])

    # Convert defaultdict to list of lists
    initial_clusters = list(temp_grouped_clusters.values())

    # --- 5. Sort the Clusters Themselves (Optional) ---
    if sort_clusters_by == "size":
        # Sort primarily by size (descending)
        # Add secondary sort key (e.g., by first element) for stable sorting if sizes are equal
        sorted_initial_clusters = sorted(
            initial_clusters, key=lambda x: (-len(x), x[0] if x else "")
        )
    else:  # 'none' or any other value
        sorted_initial_clusters = initial_clusters

    # --- 6. Sort Items WITHIN Each Cluster (Outliers First) ---
    final_clusters = []
    for cluster in sorted_initial_clusters:
        # Perform internal sorting based on avg distance to peers (outliers first)
        internally_sorted_cluster = sort_items_within_cluster(cluster)
        final_clusters.append(internally_sorted_cluster)

    return final_clusters


# Group columns using hierarchical clustering, sort clusters by size, sort items internally (outliers first)
columns = list(csv.columns.values)
final_grouped_clusters = group_columns_hierarchical_internal_sort(
    columns,
    normalized_threshold=0.80,  # Adjust threshold - might need higher for longer names
    linkage_method="average",
    sort_clusters_by="size",
)

In [ ]:
from Levenshtein import median, median_improve
from rich.columns import Columns
from rich.console import Console

console = Console()
for idx, cluster in enumerate(final_grouped_clusters):
    print("=" * 80)
    console.print(
        f"Cluster {idx + 1}",
    )
    median_string = median(cluster)
    improved_median_string = median_improve(median_string, cluster)
    median_sections = median_string.split(" ")
    median_sections = list(filter(None, median_sections))

    for idx, section in enumerate(median_sections):
        string = re.escape(section)
        matches = [re.search(rf"{string}", x) for x in cluster]
        num_matches = len([x for x in matches if x is not None])
        if num_matches == 0:
            if idx == len(median_sections) - 1:
                print()
            continue
        else:
            print("-" * 80)

        matching = [x for x in cluster if re.search(rf"{string}", x) is not None]
        not_matching = [x for x in cluster if re.search(rf"{string}", x) is None]

        if len(matching) == len(cluster):
            print(f'All columns in the cluster contain the string "{section}".')
        elif len(matching) != 0:
            if len(matching) < 5:
                print(f'{len(matching)} Columns Matching - "{section}": {matching}')
            else:
                columns_display = Columns(
                    matching,
                    expand=True,
                    equal=True,
                    title=f'{len(matching)} Columns Matching - "{section}"',
                )  # Adjust options as needed
                console.print(columns_display, justify="center")
            print("~" * 80)

        if len(not_matching) != 0:
            if len(not_matching) < 5:
                print(
                    f'{len(not_matching)} Columns Not Matching - "{section}": {not_matching}'
                )
            else:
                columns_display_nm = Columns(
                    not_matching,
                    expand=True,
                    equal=True,
                    title=f'{len(not_matching)} Columns Not Matching - "{section}"',
                )  # Adjust options as needed
                console.print(columns_display_nm, justify="center")

        if idx != len(median_sections) - 1:
            print("-" * 80)
        else:
            print()

In [ ]:
# removes the parentheses and replace spaces with underscores from the column names - optional
csv.columns = [re.sub(r"[\(\)]", "", x) for x in csv.columns]
csv.columns = [re.sub(r" ", "_", x) for x in csv.columns]
csv.columns

In [ ]:
# isolate phate dimenionality reduction coordinates columns
phate_cols = csv.filter(regex="PHATE").columns
phate_sc_df = csv.loc[:, phate_cols].copy()
csv.drop(phate_cols, axis=1, inplace=True)
phate_sc_df.head()

In [ ]:
# isolate all columns that are strings that will be converted into the obs for the anndata object
# also make sure to remove all cols not relevant for analysis. i.e. WellID, etc.
sc_obs_df = csv.select_dtypes(include=["object"])
sc_obs_df.columns

In [ ]:
# isolate the feature columns - ensure that all columns included in this dataframe are numeric and relevant to the analysis
sc_feat_df = csv.select_dtypes(exclude=["object"])
sc_feat_df.columns

# Preprocessing of AnnData Object

In [ ]:
# convert the dataframe into an anndata object
adata = ad.AnnData(
    X=sc_feat_df.values,
    obs=sc_obs_df.copy(),
    var=pd.DataFrame(index=sc_feat_df.columns),
)
adata

In [ ]:
## Let's inspect some example clusters
# print(f"Total clusters: {len(final_grouped_clusters)}\n")

## Show first few clusters
# for idx in [0, 1, 2, 9]:  # Including cluster 9 which has the median features
# cluster = final_grouped_clusters[idx]
# print(f"{'='*80}")
# print(f"Cluster {idx+1} ({len(cluster)} items):")
# print(cluster[:10] if len(cluster) > 10 else cluster)  # Show first 10 or all
# print()

# Pattern Extraction Layer

The idea: Keep the clustering (it works great!), but add an automatic pattern detector that extracts common suffixes/infixes from each cluster. This makes it easier to select which types of features you want.

In [ ]:
import re
from collections import Counter


def extract_common_patterns(cluster, min_frequency=0.7):
    """
    Extract common patterns (suffixes, prefixes, infixes) from a cluster of column names.

    Args:
        cluster: List of column names
        min_frequency: Minimum fraction of cluster members that must contain pattern

    Returns:
        dict with 'suffixes', 'prefixes', and 'common_substrings'
    """
    if not cluster:
        return {}

    # Extract potential suffixes (last part after underscore or space)
    suffixes = []
    prefixes = []

    for col in cluster:
        # Get suffix patterns (everything after last underscore, or last 2 parts)
        parts = re.split(r"[_\s]+", col)
        if len(parts) >= 2:
            suffixes.append("_".join(parts[-2:]))  # Last 2 parts
            suffixes.append(parts[-1])  # Just last part
            prefixes.append(parts[0])  # First part

    # Count occurrences
    suffix_counts = Counter(suffixes)
    prefix_counts = Counter(prefixes)

    min_count = len(cluster) * min_frequency

    # Filter patterns that appear frequently enough
    common_suffixes = [
        pattern
        for pattern, count in suffix_counts.items()
        if count >= min_count and len(pattern) > 2
    ]
    common_prefixes = [
        pattern
        for pattern, count in prefix_counts.items()
        if count >= min_count and len(pattern) > 2
    ]

    # Find common substrings using the median string approach
    common_substrings = []
    for col in cluster:
        # Extract meaningful parts (longer than 3 chars)
        parts = [p for p in re.split(r"[_\s]+", col) if len(p) > 3]
        common_substrings.extend(parts)

    substring_counts = Counter(common_substrings)
    common_substrings = [
        pattern for pattern, count in substring_counts.items() if count >= min_count
    ]

    return {
        "suffixes": sorted(set(common_suffixes), key=len, reverse=True),
        "prefixes": sorted(set(common_prefixes), key=len, reverse=True),
        "substrings": sorted(set(common_substrings), key=len, reverse=True),
    }


# Extract patterns from each cluster
cluster_patterns = []
for idx, cluster in enumerate(final_grouped_clusters):
    patterns = extract_common_patterns(cluster, min_frequency=0.7)
    cluster_patterns.append(
        {
            "cluster_idx": idx,
            "size": len(cluster),
            "patterns": patterns,
            "sample_columns": cluster[:3],  # Show first 3 as examples
        }
    )

# Display pattern summary
print("=" * 100)
print("PATTERN EXTRACTION RESULTS")
print("=" * 100)
print(f"\nFound {len(cluster_patterns)} clusters with the following patterns:\n")

for cp in cluster_patterns[:15]:  # Show first 15 clusters
    print(f"Cluster {cp['cluster_idx'] + 1} ({cp['size']} columns)")
    print(f"  Examples: {cp['sample_columns']}")

    if cp["patterns"]["suffixes"]:
        print(f"  Common suffixes: {cp['patterns']['suffixes'][:3]}")
    if cp["patterns"]["prefixes"]:
        print(f"  Common prefixes: {cp['patterns']['prefixes'][:3]}")
    if cp["patterns"]["substrings"]:
        print(f"  Common substrings: {cp['patterns']['substrings'][:5]}")
    print()

## How the Pattern Extraction Layer Works:

**The clustering stays exactly the same** - it groups similar column names using Levenshtein distance.

**The pattern extraction layer adds on top:**
1. **Analyzes each cluster** to find common patterns (suffixes, prefixes, substrings)
2. **Identifies naming conventions** like `cell_median`, `nuc_median`, `cyto_median`, `PM_median`, `PN_median`
3. **Lets you select patterns** to keep as features vs. move to other locations

This gives you a **two-level view**:
- **Level 1 (Clustering):** Groups columns that look similar
- **Level 2 (Pattern Extraction):** Tells you *what makes them similar* so you can make decisions

For example:
- Cluster 2 has pattern `cell_median` → You can say "I want all `cell_median` features"
- Cluster 4 has pattern `nuc_median` → You can say "I want all `nuc_median` features"  
- Cluster 1 has pattern `PM_median`/`PN_median` → You can say "These go to obs/layers, not features"

In [ ]:
def filter_columns_by_patterns(
    columns,
    clusters,
    feature_patterns=None,
    obs_patterns=None,
    layer_patterns=None,
    exclude_patterns=None,
):
    """
    Categorize columns based on pattern matching.

    The clustering helps group similar columns, then you specify which patterns
    you want in each category.

    Args:
        columns: List of all column names
        clusters: The clustered groups from hierarchical clustering
        feature_patterns: List of patterns to keep as features (e.g., ['cell_median', 'nuc_median'])
        obs_patterns: List of patterns for observation metadata
        layer_patterns: List of patterns for AnnData layers
        exclude_patterns: List of patterns to exclude entirely

    Returns:
        dict with keys 'features', 'obs', 'layers', 'excluded', 'uncategorized'
    """
    if feature_patterns is None:
        feature_patterns = []
    if obs_patterns is None:
        obs_patterns = []
    if layer_patterns is None:
        layer_patterns = []
    if exclude_patterns is None:
        exclude_patterns = []

    result = {
        "features": [],
        "obs": [],
        "layers": [],
        "excluded": [],
        "uncategorized": [],
    }

    for col in columns:
        categorized = False

        # Check exclusions first
        if any(pattern in col for pattern in exclude_patterns):
            result["excluded"].append(col)
            categorized = True
            continue

        # Check feature patterns
        if any(pattern in col for pattern in feature_patterns):
            result["features"].append(col)
            categorized = True
            continue

        # Check obs patterns
        if any(pattern in col for pattern in obs_patterns):
            result["obs"].append(col)
            categorized = True
            continue

        # Check layer patterns
        if any(pattern in col for pattern in layer_patterns):
            result["layers"].append(col)
            categorized = True
            continue

        # If not categorized, add to uncategorized
        if not categorized:
            result["uncategorized"].append(col)

    return result


# Example: Let's say you want cell_median and nuc_median as features
example_categorization = filter_columns_by_patterns(
    columns=list(csv.columns),
    clusters=final_grouped_clusters,
    feature_patterns=["cell_median", "nuc_median"],  # These become your X matrix
    layer_patterns=[
        "PM_median",
        "PN_median",
        "cyto_median",
    ],  # These could go to layers
    obs_patterns=["phase", "age"],  # Metadata
    exclude_patterns=["PHATE", "area", "shape"],  # Don't want these
)

print("CATEGORIZATION RESULTS:")
print("=" * 80)
print(f"\nFeatures (X matrix): {len(example_categorization['features'])} columns")
print(f"  Examples: {example_categorization['features'][:5]}")
print(f"\nLayers: {len(example_categorization['layers'])} columns")
print(f"  Examples: {example_categorization['layers'][:5]}")
print(f"\nObs (metadata): {len(example_categorization['obs'])} columns")
print(f"  Examples: {example_categorization['obs'][:5]}")
print(f"\nExcluded: {len(example_categorization['excluded'])} columns")
print(f"  Examples: {example_categorization['excluded'][:5]}")
print(f"\nUncategorized: {len(example_categorization['uncategorized'])} columns")
print(f"  Examples: {example_categorization['uncategorized'][:10]}")

## Summary: How it all works together

```
Step 1: CLUSTERING (existing, unchanged)
├─ Takes all column names
├─ Groups similar ones using Levenshtein distance  
└─ Creates clusters like: [cycD1_nuc_median, cycB1_nuc_median, ...] 

Step 2: PATTERN EXTRACTION (new layer)
├─ Analyzes each cluster
├─ Finds common patterns: "nuc_median", "cell_median", "PM_median", etc.
└─ Shows you what makes each cluster similar

Step 3: USER SELECTION (you decide)
├─ You specify: feature_patterns=['cell_median', 'nuc_median']
├─ You specify: layer_patterns=['PM_median', 'PN_median', 'cyto_median']
└─ You specify: exclude_patterns=['area', 'shape']

Step 4: AUTOMATIC FILTERING (new function)
├─ Categorizes ALL columns based on your patterns
├─ Returns: features, obs, layers, excluded, uncategorized
└─ Ready to build your AnnData object!
```

**Key insight:** The clustering doesn't change - it's still doing the heavy lifting of grouping similar columns. The pattern extraction just makes it easier for you to say "I want all columns like *this*" without listing them one by one.

## Practical Usage - Replace your current filtering

Instead of manually using `select_dtypes()` and `filter(regex=...)`, you can now do:

In [ ]:
# YOUR DESIRED WORKFLOW: Select the patterns you want

# 1. Cluster the columns (already done above)
# 2. Decide which patterns are features vs. other categories
categorized = filter_columns_by_patterns(
    columns=list(csv.columns),
    clusters=final_grouped_clusters,
    # YOU DECIDE: Which patterns go where
    feature_patterns=["cell_median", "nuc_median"],  # Main features for analysis
    layer_patterns=[
        "cyto_median",
        "PM_median",
        "PN_median",
    ],  # Alternative measurements
    obs_patterns=["phase", "age"],  # Metadata/annotations
    exclude_patterns=["PHATE", "area", "shape", "density"],  # Not needed
)

# 3. Use the categorization to split your dataframe
features_df = csv[categorized["features"]].copy()
layers_dict = {
    "cyto_median": csv[[c for c in categorized["layers"] if "cyto_median" in c]],
    "PM_median": csv[[c for c in categorized["layers"] if "PM_median" in c]],
    "PN_median": csv[[c for c in categorized["layers"] if "PN_median" in c]],
}
obs_df = csv[categorized["obs"]].copy()

print("✓ Features DataFrame:", features_df.shape)
print("✓ Obs DataFrame:", obs_df.shape)
print("✓ Layers:")
for layer_name, layer_df in layers_dict.items():
    print(f"  - {layer_name}: {layer_df.shape}")

print(f"\n✓ Uncategorized: {len(categorized['uncategorized'])} columns")
print(f"  These need decisions: {categorized['uncategorized'][:5]}...")

In [ ]:
# Show uncategorized columns organized by their clusters
# This helps you decide what to do with them

print("\nUNCATEGORIZED COLUMNS BY CLUSTER:")
print("=" * 80)

uncategorized_set = set(categorized["uncategorized"])

for idx, cluster in enumerate(final_grouped_clusters):
    # Check if this cluster has any uncategorized columns
    uncategorized_in_cluster = [col for col in cluster if col in uncategorized_set]

    if uncategorized_in_cluster:
        patterns = extract_common_patterns(cluster, min_frequency=0.5)
        print(
            f"\nCluster {idx + 1} ({len(uncategorized_in_cluster)} uncategorized / {len(cluster)} total)"
        )

        if patterns["suffixes"]:
            print(f"  Patterns: {', '.join(patterns['suffixes'][:3])}")
        elif patterns["substrings"]:
            print(f"  Common words: {', '.join(patterns['substrings'][:3])}")

        print(f"  Columns: {uncategorized_in_cluster[:5]}")
        if len(uncategorized_in_cluster) > 5:
            print(f"           ... and {len(uncategorized_in_cluster) - 5} more")

## Can this be done unsupervised? Let's explore options:

### Option 1: **Rule-based heuristics** (No LLM needed!)
The clustering + pattern extraction is already doing most of the work. You could add simple rules:
- Anything with `_median` at the end → likely a feature
- Anything with just 1-2 unique values → likely obs/metadata
- Anything with `phospho`, `area`, `shape` → you decide per dataset type

### Option 2: **Semi-supervised with defaults**
Create a config file or defaults based on common patterns:
```python
default_feature_patterns = ['nuc_median', 'cell_median', 'cyto_median', 'IntegratedIntensity']
default_obs_patterns = ['phase', 'age', 'condition', 'treatment']
default_exclude_patterns = ['PHATE', 'UMAP', 'tSNE']
```

### Option 3: **Interactive widget** (Best of both worlds)
Use ipywidgets to let users click checkboxes for each cluster pattern. Still fast, but gives control.

### Option 4: **Learn from previous datasets**
If you're processing multiple similar datasets, save the pattern choices and reuse them.

**My recommendation:** Don't avoid this feature! The clustering is valuable. Instead, think of it as a **one-time configuration per dataset type** rather than needing decisions every time.

In [ ]:
def auto_categorize_columns_heuristic(columns, clusters):
    """
    Automatic categorization using simple heuristics - NO LLM needed!

    This uses rules based on common naming patterns in biology datasets.
    You can customize these rules for your specific use case.
    """

    result = {
        "features": [],
        "obs": [],
        "layers": [],
        "excluded": [],
        "uncategorized": [],
    }

    for col in columns:
        col_lower = col.lower()

        # Rule 1: PHATE/UMAP/tSNE are dimensionality reductions -> exclude from features
        if any(x in col_lower for x in ["phate", "umap", "tsne", "pca"]) or any(
            x in col_lower for x in ["area", "shape", "formfactor", "density"]
        ):
            result["excluded"].append(col)

        # Rule 3: Phase, age, condition, treatment -> metadata (obs)
        elif any(
            x in col_lower
            for x in [
                "phase",
                "age",
                "condition",
                "treatment",
                "timepoint",
                "replicate",
            ]
        ):
            result["obs"].append(col)

        # Rule 4: Main feature patterns - nucleus and cell medians are typically primary features
        elif any(pattern in col for pattern in ["nuc_median", "cell_median"]):
            result["features"].append(col)

        # Rule 5: Cytoplasm medians could be layers (alternative measurements)
        elif "cyto_median" in col or any(
            x in col for x in ["PM_median", "PN_median", "_PM", "_PN"]
        ):
            result["layers"].append(col)

        # Rule 7: Phospho/total ratios -> often features
        elif (
            "phospho/total" in col
            or any(x in col for x in ["Int_Intg", "DNA_content", "foci"])
            or "_over_" in col
            or ("/" in col and "phospho" not in col)
        ):
            result["features"].append(col)

        # Everything else -> uncategorized (needs manual review)
        else:
            result["uncategorized"].append(col)

    return result


# Test automatic categorization
auto_categorized = auto_categorize_columns_heuristic(
    columns=list(csv.columns), clusters=final_grouped_clusters
)

print("AUTOMATIC HEURISTIC CATEGORIZATION:")
print("=" * 80)
print(f"\n✓ Features (X matrix): {len(auto_categorized['features'])} columns")
print(f"  Examples: {auto_categorized['features'][:8]}")
print(f"\n✓ Layers: {len(auto_categorized['layers'])} columns")
print(f"  Examples: {auto_categorized['layers'][:5]}")
print(f"\n✓ Obs (metadata): {len(auto_categorized['obs'])} columns")
print(f"  All obs: {auto_categorized['obs']}")
print(f"\n✓ Excluded: {len(auto_categorized['excluded'])} columns")
print(f"  Examples: {auto_categorized['excluded'][:5]}")
print(
    f"\n⚠ Uncategorized: {len(auto_categorized['uncategorized'])} columns (need rules or manual review)"
)
print(f"  Examples: {auto_categorized['uncategorized'][:10]}")

### 🎉 Success! The heuristic approach categorized everything!

**Key insights:**
1. **No LLM needed** - Simple pattern matching rules work great
2. **Clustering helps validate** - You can see which rules capture which clusters
3. **Rules are interpretable** - Easy to debug and customize
4. **Fast and deterministic** - Same input = same output every time

**This could easily become a pipeline function:**
```python
def prepare_anndata_from_csv(csv_path, rules='default'):
    # 1. Load CSV
    # 2. Auto-cluster columns
    # 3. Auto-categorize using heuristics
    # 4. Build AnnData object
    # 5. Return ready-to-use adata
```

**When you might still want manual review:**
- Novel dataset types with new naming conventions
- Mixed experiments with different measurement types
- Custom features that don't follow standard patterns

**Bottom line:** Don't avoid this feature! It's actually perfect for automation with simple rules.

In [ ]:
def csv_to_anndata_auto(
    df,
    feature_rules=None,
    layer_rules=None,
    obs_rules=None,
    exclude_rules=None,
    clustering_threshold=0.80,
    return_diagnostics=False,
):
    """
    Automated pipeline: CSV → AnnData with minimal user input.

    Steps:
    1. Cluster similar column names
    2. Auto-categorize columns using heuristic rules
    3. Build AnnData object with proper structure

    Args:
        df: Input DataFrame
        feature_rules: Custom rules for features (optional, uses defaults)
        layer_rules: Custom rules for layers (optional, uses defaults)
        obs_rules: Custom rules for obs (optional, uses defaults)
        exclude_rules: Custom rules for exclusions (optional, uses defaults)
        clustering_threshold: Levenshtein distance threshold for clustering
        return_diagnostics: If True, returns (adata, diagnostics_dict)

    Returns:
        AnnData object (or tuple with diagnostics if requested)
    """

    # Default rules (can be overridden)
    if feature_rules is None:
        feature_rules = [
            lambda col: "nuc_median" in col,
            lambda col: "cell_median" in col,
            lambda col: "phospho/total" in col,
            lambda col: "Int_Intg" in col,
            lambda col: "_over_" in col and "phospho" not in col,
            lambda col: "DNA_content" in col,
            lambda col: "foci" in col,
        ]

    if layer_rules is None:
        layer_rules = [
            lambda col: "cyto_median" in col,
            lambda col: "PM_median" in col or "_PM" in col,
            lambda col: "PN_median" in col or "_PN" in col,
        ]

    if obs_rules is None:
        obs_rules = [
            lambda col: (
                col.lower() in ["phase", "age", "condition", "treatment", "timepoint"]
            ),
        ]

    if exclude_rules is None:
        exclude_rules = [
            lambda col: any(x in col.lower() for x in ["phate", "umap", "tsne", "pca"]),
            lambda col: any(
                x in col.lower() for x in ["area", "shape", "formfactor", "density"]
            ),
        ]

    # Step 1: Cluster columns
    columns = list(df.columns)
    clusters = group_columns_hierarchical_internal_sort(
        columns,
        normalized_threshold=clustering_threshold,
        linkage_method="average",
        sort_clusters_by="size",
    )

    # Step 2: Categorize columns
    categorized = {
        "features": [],
        "layers": [],
        "obs": [],
        "excluded": [],
        "uncategorized": [],
    }

    for col in columns:
        # Check each rule category in order
        if any(rule(col) for rule in exclude_rules):
            categorized["excluded"].append(col)
        elif any(rule(col) for rule in obs_rules):
            categorized["obs"].append(col)
        elif any(rule(col) for rule in feature_rules):
            categorized["features"].append(col)
        elif any(rule(col) for rule in layer_rules):
            categorized["layers"].append(col)
        else:
            categorized["uncategorized"].append(col)

    # Step 3: Build AnnData
    # Features go in X
    X = df[categorized["features"]].values if categorized["features"] else None

    # Obs goes in obs
    obs_df = df[categorized["obs"]].copy() if categorized["obs"] else None

    # Var metadata
    var_df = (
        pd.DataFrame(index=categorized["features"]) if categorized["features"] else None
    )

    # Create base AnnData
    adata = ad.AnnData(X=X, obs=obs_df, var=var_df)

    # Add layers if any
    if categorized["layers"]:
        # Group layers by pattern
        layer_patterns = {
            "cyto_median": [c for c in categorized["layers"] if "cyto_median" in c],
            "PM_median": [
                c for c in categorized["layers"] if "PM_median" in c or "_PM" in c
            ],
            "PN_median": [
                c for c in categorized["layers"] if "PN_median" in c or "_PN" in c
            ],
        }

        for layer_name, layer_cols in layer_patterns.items():
            if layer_cols:
                adata.layers[layer_name] = df[layer_cols].values

    # Return with diagnostics if requested
    if return_diagnostics:
        diagnostics = {
            "clusters": clusters,
            "categorization": categorized,
            "n_features": len(categorized["features"]),
            "n_layers": len(categorized["layers"]),
            "n_obs_columns": len(categorized["obs"]),
            "n_excluded": len(categorized["excluded"]),
            "n_uncategorized": len(categorized["uncategorized"]),
        }
        return adata, diagnostics

    return adata


# Test the automated pipeline
adata_auto, diagnostics = csv_to_anndata_auto(csv, return_diagnostics=True)

print("AUTOMATED PIPELINE RESULTS:")
print("=" * 80)
print(f"\n✓ AnnData object created: {adata_auto}")
print("\n✓ Diagnostics:")
print(f"  - Features (X): {diagnostics['n_features']} columns")
print(f"  - Layers: {diagnostics['n_layers']} columns")
print(f"  - Obs metadata: {diagnostics['n_obs_columns']} columns")
print(f"  - Excluded: {diagnostics['n_excluded']} columns")
print(f"  - Uncategorized: {diagnostics['n_uncategorized']} columns")
print(f"\n✓ Available layers: {list(adata_auto.layers.keys())}")
print(f"\n✓ Obs columns: {list(adata_auto.obs.columns)}")